# 02 · LoRA fine-tune `nvidia/Nemotron-Mini-4B-Instruct`

**Switch this runtime to a T4 GPU** (Runtime → Change runtime type → T4 GPU) before running.

Loads the base model in 4-bit, attaches a LoRA adapter via `peft`, trains with `trl.SFTTrainer` on the split saved by `01_prepare_dataset.ipynb`, then pushes just the adapter to your own Hugging Face namespace so it can be loaded on top of the base model at serve time by vLLM or SGLang.

In [ ]:
# Capped below their next major version: transformers 5.x and trl 1.x
# introduced breaking internal changes (trl added a "chunked loss" code
# path that doesn't support every model/PEFT combination, among other
# renames) relative to what this notebook is written against. Capping
# lands on the last release before each break instead of chasing
# individual incompatibilities one at a time. peft/accelerate/bitsandbytes
# haven't shown any version-specific issues here, so they're left
# unbounded.
!pip install -q "transformers>=4.44,<5.0" "peft>=0.12" "trl>=0.9,<1.0" "accelerate>=0.33" "bitsandbytes>=0.43" datasets huggingface_hub

In [ ]:
# This is a separate Colab runtime from 01_prepare_dataset.ipynb, so mount
# Drive again to reach the split saved there.
from google.colab import drive

drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/nemotron-rag-serving-lab"

In [ ]:
from huggingface_hub import login

# Needs a WRITE-scoped token this time, since this notebook pushes the adapter
# to your own namespace: https://huggingface.co/settings/tokens
login()

In [ ]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL = "nvidia/Nemotron-Mini-4B-Instruct"

# float16, not bfloat16: the free-tier T4 GPU (Turing architecture) has no
# native bf16 support, and running bf16 ops on it silently falls back to a
# much slower path (~30-50x) instead of erroring. If you're running this on
# an Ampere+ GPU (A100, L4, RTX 30xx/40xx) you can switch these back to
# torch.bfloat16 below for a small quality/stability edge.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,  # match bnb_4bit_compute_dtype above (may warn "use dtype instead" on newer transformers -- harmless)
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)

# Belt-and-suspenders: this checkpoint keeps some parameters/buffers in
# bfloat16 regardless of the dtype requested at load time above (observed
# on more than one transformers/trl version, so it's not just a version
# thing) -- and fp16 training's GradScaler can't unscale a bfloat16
# gradient ("_amp_foreach_non_finite_check_and_unscale_cuda ... not
# implemented for 'BFloat16'"). Forcing everything to float16 here,
# after the model + LoRA adapter both exist, sidesteps it directly
# instead of hunting for which specific submodule is responsible.
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
for _, buf in model.named_buffers():
    if buf.dtype == torch.bfloat16:
        buf.data = buf.data.to(torch.float16)

model.print_trainable_parameters()

In [ ]:
train_ds = load_from_disk(f"{SAVE_DIR}/daring_anteater_train")
eval_ds = load_from_disk(f"{SAVE_DIR}/daring_anteater_eval")
print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")

In [ ]:
import inspect

from trl import SFTConfig, SFTTrainer

sft_config_kwargs = dict(
    output_dir="/content/nemotron-mini-4b-daring-anteater-lora",
    # Shorter sequences (see max_len_field below) free up enough memory to
    # afford a real per-device batch instead of 1, which uses the GPU far
    # more efficiently than launching 16 size-1 kernels per step.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # effective batch size 16, same as before
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="epoch",
    # bf16, not fp16, at the Trainer level: this checkpoint's own modeling
    # code hardcodes a bfloat16 cast somewhere internally no matter what
    # dtype we load/cast everything else to, and fp16 training's GradScaler
    # can't unscale a bfloat16 gradient
    # ("_amp_foreach_non_finite_check_and_unscale_cuda ... not implemented
    # for 'BFloat16'"). bf16 doesn't use a GradScaler at all, so that crash
    # can't happen -- though in practice bf16 here ended up applying much
    # more broadly than just that one op (autocast doesn't only touch the
    # spot that needed it), which is the real reason this configuration
    # keeps its total step count and per-step cost down rather than relying
    # on bf16 being cheap on a T4.
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="text",
    report_to="none",
)
# trl renamed this field across versions (max_seq_length -> max_length).
# Set whichever name this installed version actually declares instead of
# hardcoding one and risking a TypeError on the other.
max_len_field = "max_length" if "max_length" in SFTConfig.__dataclass_fields__ else "max_seq_length"
sft_config_kwargs[max_len_field] = 256
sft_config = SFTConfig(**sft_config_kwargs)

trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
)
# Same story for the tokenizer/processor argument name (tokenizer -> processing_class).
tok_param = "processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer"
trainer_kwargs[tok_param] = tokenizer
trainer = SFTTrainer(**trainer_kwargs)

trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

In [ ]:
# Push ONLY the adapter (a few hundred MB), not a merged copy of the base model.
# Replace <your-hf-username> below.
HUB_REPO = "<your-hf-username>/nemotron-mini-4b-daring-anteater-lora"

trainer.model.push_to_hub(HUB_REPO)
tokenizer.push_to_hub(HUB_REPO)
print(f"Adapter pushed to https://huggingface.co/{HUB_REPO}")
print("Use this repo id as --lora-modules / --lora-path in the vLLM and SGLang serve scripts.")